# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [61]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [62]:
# Check available columns

df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Feature Notes

The feature vector was designed using only information that is expected to be available before the prediction time. Private identifiers, system-generated fields, and potentially future-leaking features were excluded.

| Feature | Meaning | Data Type | Missing Value Handling | Available When? |
|---|---|---|---|---|
| search_volume | Estimated search demand for the topic/query | Numeric | Fill with 0 | Before prediction |
| competition | Competition score for the keyword/topic | Numeric | Fill with 0 | Before prediction |
| competition_level | Category representing competition intensity | Categorical | One-hot encoding + handle missing | Before prediction |
| cpc | Cost per click value indicating commercial value | Numeric | Fill with 0 | Before prediction |
| content_type | Type/category of content | Categorical | One-hot encoding + handle missing | Before prediction |
| main_intent | Search intent category of the content | Categorical | One-hot encoding + handle missing | Before prediction |
| word_count | Number of words in the content | Numeric | Fill with 0 | Before prediction |
| char_count | Number of characters in the content | Numeric | Fill with 0 | Before prediction |
| content_age_days | Age of content in days | Numeric | Fill with 0 | Before prediction |
| impressions_prev_30d | Search impressions from previous 30-day window | Numeric | Fill with 0 | Available before prediction |
| clicks_prev_30d | Clicks from previous 30-day window | Numeric | Fill with 0 | Available before prediction |
| sessions_prev_30d | User sessions from previous 30-day window | Numeric | Fill with 0 | Available before prediction |
| trend_direction | Direction of historical performance trend | Categorical | One-hot encoding + handle missing | Before prediction |
| trend_pct | Percentage change in historical performance | Numeric | Fill with 0 | Before prediction |

## Categorical Handling

Categorical variables were converted into numerical representation using one-hot encoding. This allows machine learning models to process non-numeric information while avoiding artificial ordering between categories.

## Prediction-Time Availability

All selected features represent historical information available before making a prediction. Features containing future information or overlapping evaluation periods were excluded to prevent data leakage.

In [63]:
print(df.shape)

print(df.columns.tolist())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [64]:
for i, col in enumerate(df.columns):
    print(i, col)

0 content_id
1 client_id
2 search_volume
3 competition
4 competition_level
5 cpc
6 content_type
7 main_intent
8 word_count
9 char_count
10 provider_used
11 model_used
12 impressions_90d
13 clicks_90d
14 pageviews_90d
15 sessions_90d
16 users_90d
17 engaged_sessions_90d
18 ai_sessions_90d
19 scroll_events_90d
20 days_with_impressions
21 days_with_sessions
22 impressions_last_30d
23 clicks_last_30d
24 sessions_last_30d
25 impressions_prev_30d
26 clicks_prev_30d
27 sessions_prev_30d
28 content_age_days
29 age_tier
30 age_tier_order
31 days_since_last_update
32 freshness_tier
33 word_count_tier
34 char_count_tier
35 ctr
36 avg_position
37 engagement_rate
38 scroll_rate
39 ai_traffic_pct
40 impression_tier
41 position_tier
42 trend_direction
43 trend_pct


In [65]:
for col in df.columns:
    print(col)
    

content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct


In [66]:
features = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "content_age_days",
    
    # historical performance
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    
    # trend information
    "trend_direction",
    "trend_pct"
]


X = df[features].copy()

print("Feature vector shape:", X.shape)

X.head()

Feature vector shape: (30000, 14)


,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,content_age_days,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,trend_direction,trend_pct
0,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,187,987,13,9,down,-41.4
1,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,445,5915,1,2,down,-57.7
2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,141,6089,3,3,down,-60.9
3,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,463,4206,17,26,stable,-13.8
4,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,263,6452,2,9,down,-34.7


In [67]:
drop_features = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used"
]

df = df.drop(columns=drop_features)

In [68]:
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "trend_direction"
]


X = pd.get_dummies(
    X,
    columns=categorical_features,
    dummy_na=True
)


X.head()

,search_volume,competition,cpc,word_count,char_count,content_age_days,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,trend_pct,...,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_nan,trend_direction_down,trend_direction_flat,trend_direction_new,trend_direction_stable,trend_direction_up,trend_direction_nan
0,10.0,0.67,2.05,3221.0,20457.0,187,987,13,9,-41.4,...,False,False,True,False,True,False,False,False,False,False
1,90.0,0.01,0.05,2481.0,15562.0,445,5915,1,2,-57.7,...,True,False,False,False,True,False,False,False,False,False
2,0.0,0.00,0.00,3515.0,23643.0,141,6089,3,3,-60.9,...,True,False,False,False,True,False,False,False,False,False
3,10.0,0.00,0.00,NaN,NaN,463,4206,17,26,-13.8,...,False,False,False,False,False,False,False,True,False,False
4,0.0,0.00,0.00,2803.0,17469.0,263,6452,2,9,-34.7,...,True,False,False,False,True,False,False,False,False,False


In [69]:
missing = X.isnull().sum()

missing[missing > 0]

search_volume    2468
competition      2468
cpc              2468
word_count       7699
char_count       7699
trend_pct        3388
dtype: int64

In [70]:
X = X.fillna(0)

print("Remaining missing values:")
print(X.isnull().sum().sum())

Remaining missing values:
0


In [71]:
print("Rows:", X.shape[0])
print("Features:", X.shape[1])

Rows: 30000
Features: 29


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## Leakage Hunt

### 1. Private Identifiers

Excluded features:

- content_id
- client_id

Reason:

These fields uniquely identify content and clients. They do not represent meaningful patterns and can cause privacy risks or memorization by the model.

---

### 2. Product/System Leakage

Excluded features:

- provider_used
- model_used

Reason:

These fields describe internal content generation decisions. They may allow the model to learn system behavior instead of real content performance patterns.

---

### 3. Future Window Leakage

Potential leakage features:

- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d

Reason:

These features may include information from the future prediction period. Using them can make offline evaluation unrealistically high.

---

### 4. Safe Historical Windows

Used:

- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d

Reason:

Previous 30-day metrics are available before prediction time and represent valid historical signals.

---

### 5. Causal Claims

The model features can identify patterns associated with content refresh needs. However, they do not prove that a feature directly causes traffic improvement.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Field Type | Reason |
|---|---|
| client_hash_id | Private identifier, no predictive meaning |
| content_hash_id | Identifier only |
| raw URLs | May contain sensitive information |
| raw queries | Privacy risk and memorization risk |
| future 30/90 day metrics | Future leakage |
| product flags | Business decision leakage |
| health_score | Possible target-derived metric |

In [72]:
excluded = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used"
]

for col in excluded:
    print(col, col in X.columns)

content_id False
client_id False
provider_used False
model_used False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [73]:
df.columns.tolist()

['search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']